# 53 Protected-point preservation check (KDTree)

Thesis §3.5 defines the preservation invariant as the Euclidean distance from each protected point to the nearest surviving vertex of the anonymized mesh. This notebook computes that metric for every valid subject and also reports the same distance against the *original* mesh, so that picker-precision effects (landmarks clicked in sparse regions of the triangulation) can be separated from pipeline displacement.

- `d_anon_mm` = distance from landmark to nearest vertex of the anonymized mesh.
- `d_orig_mm` = distance from landmark to nearest vertex of the original mesh.
- `delta_mm  = d_anon_mm - d_orig_mm`.

The pipeline claim is $\Delta = 0$: no surviving vertex is repositioned, so the nearest-vertex identity is preserved bit-exact. The absolute `d_anon_mm` characterizes picker precision, not pipeline error, and can exceed the 1 mm edge-length heuristic on sparse regions of the Einstar mesh (e.g. pre-auricular notch, high vertex).

**Cohort.** Iterates over the eleven valid thesis subjects: the optode-cap cohort S1--S7 (Subject 16--22) and the bare-cap cohort S8--S11 (Subject 12--15). Subject 11 is excluded (unusable raw scan). The output CSVs carry the shared `s_id` and `optode` columns so the LaTeX tables can render cohort membership directly.

Populates §4.4.1 of the thesis. Outputs (under `thesis_results_out/`):
- `preservation_check.csv` -- one row per (subject, landmark)
- `preservation_summary.csv` -- one row per subject

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, is_optode, s_id,
)

import numpy as np
import pandas as pd
from scipy.spatial import KDTree

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

ready_with_anon = [
    n for n in available_subjects() if subject_paths(n).anon_exists
]
print(f'ready_with_anon ({len(ready_with_anon)}): {ready_with_anon}')

ready_with_anon (11): [16, 17, 18, 19, 20, 21, 22, 12, 13, 14, 15]


## 1. Per-landmark table

For each subject, build a KDTree over the original-mesh vertices and another over the anonymized-mesh vertices (both in the digitized frame, same as the landmarks sidecar). Query each of the five 10-20 landmarks against both trees.

In [2]:
rows = []
for n in ready_with_anon:
    surf_orig = load_raw(n)
    surf_anon = load_anon(n)
    lm = load_landmarks(n)

    tree_orig = KDTree(np.asarray(surf_orig.mesh.vertices))
    tree_anon = KDTree(np.asarray(surf_anon.mesh.vertices))

    coords = lm.pint.dequantify().values
    labels = list(lm['label'].values)
    for label, xyz in zip(labels, coords):
        d_orig, _ = tree_orig.query(xyz, k=1)
        d_anon, _ = tree_anon.query(xyz, k=1)
        rows.append({
            's_id': s_id(n),
            'subject': n,
            'optode': is_optode(n),
            'landmark': label,
            'd_orig_mm': float(d_orig),
            'd_anon_mm': float(d_anon),
            'delta_mm': float(d_anon - d_orig),
        })
df = pd.DataFrame(rows)
if len(df):
    df = df.iloc[df['s_id'].map(lambda s: int(s[1:])).argsort(kind='stable')].reset_index(drop=True)
df.head(15)

,s_id,subject,optode,landmark,d_orig_mm,d_anon_mm,delta_mm
0,S1,16,True,Cz,0.144092,0.144092,0.0
1,S1,16,True,Iz,0.310607,0.310607,0.0
2,S1,16,True,RPA,0.284219,0.284219,0.0
3,S1,16,True,Nz,0.496908,0.496908,0.0
4,S1,16,True,LPA,0.367272,0.367272,0.0
5,S2,17,True,Cz,0.415401,0.415401,0.0
6,S2,17,True,Iz,0.172887,0.172887,0.0
7,S2,17,True,RPA,1.813508,1.813508,0.0
8,S2,17,True,Nz,0.123483,0.123483,0.0
9,S2,17,True,LPA,0.055458,0.055458,0.0


## 2. Per-subject summary

Max across the five landmarks of `d_anon_mm` (absolute) and `delta_mm` (pipeline displacement). The primary thesis claim is that `max_delta_mm` is 0 bit-exact; `max_d_anon_mm` characterizes edge length at the landmark sites.

In [3]:
summary = (
    df.groupby(['s_id', 'subject', 'optode'], as_index=False)
    .agg(
        max_d_anon_mm=('d_anon_mm', 'max'),
        max_d_orig_mm=('d_orig_mm', 'max'),
        max_abs_delta_mm=('delta_mm', lambda s: float(s.abs().max())),
    )
)
summary['pass_1mm'] = summary.max_d_anon_mm <= 1.0
summary['pass_2mm'] = summary.max_d_anon_mm <= 2.0
summary = summary.iloc[summary['s_id'].map(lambda s: int(s[1:])).argsort()].reset_index(drop=True)
summary

,s_id,subject,optode,max_d_anon_mm,max_d_orig_mm,max_abs_delta_mm,pass_1mm,pass_2mm
0,S1,16,True,0.496908,0.496908,0.0,True,True
1,S2,17,True,1.813508,1.813508,0.0,False,True
2,S3,18,True,0.317049,0.317049,0.0,True,True
3,S4,19,True,1.529812,1.529812,0.0,False,True
4,S5,20,True,0.436609,0.436609,0.0,True,True
5,S6,21,True,0.348509,0.348509,0.0,True,True
6,S7,22,True,0.332838,0.332838,0.0,True,True
7,S8,12,False,0.349641,0.349641,0.0,True,True
8,S9,13,False,0.458919,0.458919,0.0,True,True
9,S10,14,False,0.484262,0.484262,0.0,True,True


## 3. Save

In [4]:
per_landmark_out = OUT_DIR / 'preservation_check.csv'
summary_out = OUT_DIR / 'preservation_summary.csv'
df.to_csv(per_landmark_out, index=False)
summary.to_csv(summary_out, index=False)
print(f'Max |delta| across all subjects and landmarks: {df.delta_mm.abs().max():.3e} mm')
print(f'Max d_anon across all subjects: {summary.max_d_anon_mm.max():.4g} mm')
print(f'Subjects passing 1 mm tolerance: {int(summary.pass_1mm.sum())}/{len(summary)}')
print(f'Subjects passing 2 mm tolerance: {int(summary.pass_2mm.sum())}/{len(summary)}')
print(f'Wrote {per_landmark_out}')
print(f'Wrote {summary_out}')

Max |delta| across all subjects and landmarks: 0.000e+00 mm
Max d_anon across all subjects: 1.814 mm
Subjects passing 1 mm tolerance: 9/11
Subjects passing 2 mm tolerance: 11/11
Wrote thesis_results_out/preservation_check.csv
Wrote thesis_results_out/preservation_summary.csv
